# TP05 : ACP, Arbres de Décision, Méthodes d'Ensemble et Pipelines sklearn

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP05_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

Ce TP a pour objectif de combiner la **réduction de dimensionnalité par ACP** avec des **méthodes de classification supervisée** sur le jeu de données MNIST (images de chiffres manuscrits).

Nous étudierons progressivement :
- L'Analyse en Composantes Principales (ACP) : théorie, visualisation, reconstruction
- La Régression sur Composantes Principales (PCR)
- Les Arbres de Décision : théorie et application
- Les méthodes d'ensemble : Random Forest et AdaBoost
- Les Pipelines sklearn et la recherche d'hyperparamètres avec GridSearchCV

### Documentation utile
- [sklearn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [sklearn DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [sklearn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

## Organisation du TP

Ce TP contient 5 parties : 4 obligatoires + 1 bonus :

1. **ACP sur MNIST — Réduction de dimensionnalité et visualisation**
2. **Régression sur Composantes Principales (PCR)**
3. **Arbres de Décision — Classification sur MNIST**
4. **Méthodes d'ensemble — Random Forest et AdaBoost**
5. **⭐ Bonus — Pipeline complet et GridSearchCV**

## Imports globaux

Exécutez cette cellule avant de commencer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    mean_squared_error, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Chargement du jeu de données MNIST

MNIST contient **70 000 images** de chiffres manuscrits (0–9), chacune de taille **28×28 pixels** (soit 784 variables).  
Nous travaillerons sur un sous-ensemble de **10 000 exemples** pour des raisons de temps de calcul.

> 💡 Chaque ligne de `X` est une image aplatie en vecteur de 784 valeurs de pixels (0–255).

In [ ]:
# Chargement MNIST — cette cellule est fournie, exécutez-la simplement
print("Chargement de MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_full, y_full = mnist.data, mnist.target.astype(int)

# Sous-ensemble de 10 000 images pour la rapidité
idx = np.random.choice(len(X_full), size=10000, replace=False)
X, y = X_full[idx], y_full[idx]

# Normalisation (pixels dans [0,1])
X = X / 255.0

print(f"Forme de X : {X.shape}  — {X.shape[0]} images, {X.shape[1]} pixels chacune")
print(f"Classes : {np.unique(y)}")

# Visualisation de quelques exemples
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(y[i]), fontsize=8)
    ax.axis('off')
plt.suptitle("Exemples MNIST (20 premières images)", y=1.02)
plt.tight_layout()
plt.show()

---

# Partie 1 — ACP sur MNIST : Réduction de Dimensionnalité et Visualisation

## Rappel théorique

L'**Analyse en Composantes Principales (ACP)** cherche les directions de variance maximale dans les données.

Étant donné une matrice de données centrée $\tilde{X} \in \mathbb{R}^{N \times d}$, l'ACP effectue la **décomposition en valeurs propres** de la matrice de covariance :

$$\Sigma = \frac{1}{N} \tilde{X}^T \tilde{X} \in \mathbb{R}^{d \times d}$$

Les **composantes principales** sont les vecteurs propres $v_1, v_2, \ldots, v_k$ associés aux plus grandes valeurs propres $\lambda_1 \geq \lambda_2 \geq \ldots$

La **variance expliquée** par la $i$-ème composante vaut :
$$\text{VE}_i = \frac{\lambda_i}{\sum_j \lambda_j}$$

> 💡 En pratique, on n'a jamais besoin de calculer $\Sigma$ explicitement : sklearn utilise une décomposition en valeurs singulières (SVD) plus stable numériquement.

## Étape 1 — Standardisation et ACP complète

Avant toute ACP, il est crucial de **centrer** (et éventuellement réduire) les données.  
Ici les pixels sont déjà dans [0,1] mais nous appliquons quand même un `StandardScaler` pour uniformiser.

In [ ]:
# Séparation train/test — fournie
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO : instancier un StandardScaler, faire fit_transform sur X_train, transform sur X_test
# ⚠️ Ne jamais fit sur X_test : ce serait une fuite de données !
scaler = ...
X_train_sc = ...
X_test_sc  = ...

# TODO : ajuster une ACP sans réduire la dimension (n_components=None ou 784)
# pour obtenir TOUTES les composantes et leurs variances expliquées
pca_full = ...
X_train_pca_full = ...

print(f"Nombre de composantes : {pca_full.n_components_}")
print(f"Variance expliquée par les 5 premières composantes : {pca_full.explained_variance_ratio_[:5].round(3)}")

## Étape 2 — Scree plot et variance cumulée

Le **scree plot** permet de choisir visuellement le nombre de composantes à conserver.  
On cherche en général le **coude** de la courbe, ou le nombre de composantes qui expliquent 90–95% de la variance.

In [ ]:
# TODO : tracer deux graphiques côte à côte :
# - Gauche : variance expliquée par composante (les 100 premières suffisent)
# - Droite : variance cumulée, avec une ligne horizontale à 90% et 95%

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Votre code ici

plt.tight_layout()
plt.show()

# TODO : afficher le nombre de composantes nécessaires pour atteindre 90% et 95% de variance
n_90 = ...
n_95 = ...
print(f"Composantes pour 90% de variance : {n_90}")
print(f"Composantes pour 95% de variance : {n_95}")

### 📝 Observations

*Combien de composantes sont nécessaires pour capturer 90% de la variance ?  
Quel est le ratio compression obtenu par rapport aux 784 pixels d'origine ?  
Que signifie la chute rapide du scree plot pour la structure des données MNIST ?*



## Étape 3 — Visualisation des composantes principales ("eigendigits")

Chaque composante principale peut être réinterprétée comme une **image 28×28** — on les appelle les **eigendigits**.  
Elles capturent les modes de variation les plus importants dans les images de chiffres.

In [ ]:
# TODO : afficher les 20 premières composantes principales sous forme d'images 28×28
# Indice : pca_full.components_ a la forme (n_components, 784)
# Remodeler chaque composante en (28, 28) avec .reshape(28, 28)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

# Votre code ici

plt.suptitle("Les 20 premières composantes principales — 'eigendigits'", y=1.02)
plt.tight_layout()
plt.show()

## Étape 4 — Reconstruction et perte d'information

En conservant seulement $k$ composantes, on peut **reconstruire** une approximation de l'image originale.  
La qualité de la reconstruction dépend du nombre de composantes conservées.

La reconstruction s'obtient par la **transformation inverse** :
$$\hat{X} = Z_k \cdot V_k^T + \mu$$

où $Z_k$ sont les coordonnées dans l'espace réduit et $V_k$ la matrice des $k$ composantes.

In [ ]:
# TODO : pour chaque valeur de n_components dans [1, 10, 30, 50, 100, 200, 784],
# reconstruire la première image du test et l'afficher
# Indice : utiliser PCA(n_components=k).fit(X_train_sc).inverse_transform(...)

n_components_list = [1, 10, 30, 50, 100, 200, 784]
fig, axes = plt.subplots(1, len(n_components_list), figsize=(16, 3))

# Votre code ici

plt.suptitle("Reconstruction MNIST selon le nombre de composantes ACP", y=1.02)
plt.tight_layout()
plt.show()

## Étape 5 — Visualisation 2D de l'espace latent

En projetant les données sur les **2 premières composantes principales**, on peut visualiser la structure du jeu de données dans un plan 2D.

In [ ]:
# TODO : projeter X_train_sc sur 2 composantes et tracer un scatter plot
# coloré par la classe (chiffre 0-9)
# Indice : utiliser plt.scatter(..., c=y_train, cmap='tab10')

pca_2d = ...
X_train_2d = ...

plt.figure(figsize=(9, 7))

# Votre code ici

plt.colorbar(label='Chiffre')
plt.title("Projection ACP 2D — MNIST (coloré par classe)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

### 📝 Observations

*Les classes sont-elles bien séparées dans l'espace 2D ?  
Quels chiffres semblent les plus proches ? Pouvez-vous expliquer pourquoi ?  
Que nous dit cette visualisation sur la difficulté du problème de classification ?*



---

# Partie 2 — Régression sur Composantes Principales (PCR)

## Rappel théorique

La **Régression sur Composantes Principales (PCR)** combine l'ACP et la régression linéaire :

1. On projette $X$ sur les $k$ premières composantes principales : $Z_k = X V_k$
2. On effectue une régression OLS sur $Z_k$ : $\hat{y} = Z_k \hat{\beta}$

**Avantages :**
- Élimine la **multicolinéarité** entre variables (les composantes sont orthogonales)
- Réduit la dimension → moins de surapprentissage
- Le nombre de composantes $k$ devient un **hyperparamètre** à valider

> ⚠️ La PCR est une méthode de **régression** (prédiction d'une valeur continue). Ici, nous allons prédire la valeur du chiffre (0–9) comme si c'était un nombre réel, puis arrondir pour obtenir une classe. C'est une simplification pédagogique — en pratique, on préfère la classification directe.

## Étape 1 — PCR : Pipeline ACP + Régression Linéaire

Nous allons comparer la PCR pour différents nombres de composantes et mesurer l'erreur quadratique moyenne (MSE).

In [ ]:
# TODO : pour chaque k dans n_components_range, construire un Pipeline
# StandardScaler → PCA(n_components=k) → LinearRegression
# Calculer le MSE sur l'ensemble de test pour chaque k

n_components_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
mse_train_list = []
mse_test_list  = []

for k in n_components_range:
    # TODO : construire le pipeline PCR
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    ...),
        ('reg',    ...)
    ])

    # TODO : entraîner et calculer MSE train et test
    pcr.fit(...)
    mse_train = ...
    mse_test  = ...

    mse_train_list.append(mse_train)
    mse_test_list.append(mse_test)

# Tracé MSE — fourni
plt.figure(figsize=(9, 4))
plt.plot(n_components_range, mse_train_list, 'o-', label='Train MSE', color='steelblue')
plt.plot(n_components_range, mse_test_list,  's--', label='Test MSE',  color='tomato')
plt.xlabel("Nombre de composantes ACP (k)")
plt.ylabel("MSE")
plt.title("PCR — MSE selon le nombre de composantes")
plt.legend()
plt.tight_layout()
plt.show()

## Étape 2 — Sélection du nombre de composantes par validation croisée

In [ ]:
# TODO : utiliser cross_val_score (cv=5, scoring='neg_mean_squared_error')
# pour trouver le k optimal parmi [10, 30, 50, 75, 100]

k_candidates = [10, 30, 50, 75, 100]
cv_mse = []

for k in k_candidates:
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k)),
        ('reg',    LinearRegression())
    ])
    # TODO : calculer le score de validation croisée et stocker le MSE moyen
    scores = ...
    cv_mse.append(...)

best_k = k_candidates[np.argmin(cv_mse)]
print(f"Meilleur k (CV) : {best_k}  —  MSE CV = {min(cv_mse):.4f}")

# Tracé CV
plt.figure(figsize=(7, 4))
plt.plot(k_candidates, cv_mse, 'o-', color='purple')
plt.axvline(best_k, color='red', linestyle='--', label=f'k optimal = {best_k}')
plt.xlabel("Nombre de composantes")
plt.ylabel("MSE (validation croisée 5-fold)")
plt.title("Sélection de k par validation croisée")
plt.legend()
plt.tight_layout()
plt.show()

## Étape 3 — Comparaison OLS brut vs PCR

In [ ]:
# TODO : comparer la régression OLS sur les pixels bruts (784 variables)
# avec la PCR au k optimal
# Afficher le MSE test et l'accuracy test (en arrondissant les prédictions)

# OLS brut
ols = Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())])
# Votre code ici

# PCR optimal
pcr_best = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=best_k)),
    ('reg',    LinearRegression())
])
# Votre code ici

print("\n=== Comparaison OLS vs PCR ===")
print(f"OLS brut   — MSE test : ...  | Accuracy (arrondi) : ...")
print(f"PCR (k={best_k}) — MSE test : ...  | Accuracy (arrondi) : ...")

### 📝 Observations

*La PCR améliore-t-elle les performances par rapport à l'OLS sur les pixels bruts ?  
Pourquoi la régression linéaire n'est-elle pas idéale pour la classification ?  
Qu'aurait-on intérêt à utiliser à la place pour une vraie tâche de classification ?*



---

# Partie 3 — Arbres de Décision : Classification sur MNIST

## Rappel théorique

Un **arbre de décision** partitionne récursivement l'espace des features en posant des questions binaires du type $x_j \leq t$.  
À chaque nœud, on choisit la coupure qui maximise le **gain d'information** :

$$\text{Gain}(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|} H(S_v)$$

avec l'**entropie** $H(S) = -\sum_k p_k \log_2 p_k$ ou l'**indice de Gini** $G(S) = 1 - \sum_k p_k^2$.

**Hyperparamètres clés :**
- `max_depth` : profondeur maximale de l'arbre
- `min_samples_split` : nombre minimum d'exemples pour diviser un nœud
- `criterion` : `'gini'` ou `'entropy'`

## Étape 1 — Calcul manuel de l'entropie et du gain d'information

Avant d'utiliser sklearn, calculons ces métriques à la main sur un exemple jouet.

In [ ]:
def entropy(y):
    """Entropie de Shannon d'un vecteur d'étiquettes y."""
    # TODO : calculer les proportions de chaque classe
    # Indice : np.unique(y, return_counts=True)
    # Rappel : H = -sum(p * log2(p)) — ignorer les p = 0
    pass

def gini(y):
    """Indice de Gini d'un vecteur d'étiquettes y."""
    # TODO : G = 1 - sum(p_k^2)
    pass

def information_gain(y_parent, y_left, y_right):
    """Gain d'information d'une coupure."""
    # TODO : IG = H(parent) - (|left|/|total|)*H(left) - (|right|/|total|)*H(right)
    pass

# Test sur un exemple jouet
y_parent = np.array([0,0,0,0,1,1,1,1])  # 4 de classe 0, 4 de classe 1 → entropie max
y_left   = np.array([0,0,0,1])           # après coupure gauche
y_right  = np.array([0,1,1,1])           # après coupure droite

print(f"Entropie parent  : {entropy(y_parent):.4f}  (attendu ≈ 1.0)")
print(f"Gini parent      : {gini(y_parent):.4f}  (attendu = 0.5)")
print(f"Gain information : {information_gain(y_parent, y_left, y_right):.4f}")

## Étape 2 — Premier arbre sur MNIST (pixels bruts)

> 💡 Nous travaillons sur un sous-ensemble à 2 chiffres (0 et 1) pour pouvoir visualiser l'arbre.

In [ ]:
# Sous-ensemble binaire 0 vs 1 — fourni
mask_train = np.isin(y_train, [0, 1])
mask_test  = np.isin(y_test,  [0, 1])
Xb_train, yb_train = X_train_sc[mask_train], y_train[mask_train]
Xb_test,  yb_test  = X_test_sc[mask_test],   y_test[mask_test]

# TODO : entraîner un DecisionTreeClassifier avec max_depth=3, criterion='gini'
tree_small = ...

# Visualisation de l'arbre — fournie
fig, ax = plt.subplots(figsize=(20, 6))
plot_tree(tree_small, max_depth=3, filled=True, feature_names=[f'px_{i}' for i in range(784)],
          class_names=['0','1'], ax=ax, fontsize=8)
plt.title("Arbre de décision — MNIST binaire (0 vs 1), max_depth=3")
plt.tight_layout()
plt.show()

print(f"Accuracy train : {tree_small.score(Xb_train, yb_train):.4f}")
print(f"Accuracy test  : {tree_small.score(Xb_test, yb_test):.4f}")

## Étape 3 — Arbre sur 10 classes (MNIST complet) et analyse du surapprentissage

In [ ]:
# TODO : entraîner des arbres pour max_depth dans [1, 3, 5, 8, 10, 15, None]
# Tracer les courbes d'accuracy train et test

depths = [1, 3, 5, 8, 10, 15, None]
acc_train_tree = []
acc_test_tree  = []

for d in depths:
    # Votre code ici
    pass

depths_plot = [d if d is not None else 20 for d in depths]
labels = [str(d) if d is not None else 'None\n(complet)' for d in depths]

plt.figure(figsize=(9, 4))
plt.plot(depths_plot, acc_train_tree, 'o-', label='Train', color='steelblue')
plt.plot(depths_plot, acc_test_tree,  's--', label='Test',  color='tomato')
plt.xticks(depths_plot, labels)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Arbre de décision — Surapprentissage selon la profondeur")
plt.legend()
plt.tight_layout()
plt.show()

## Étape 4 — Arbre sur features ACP vs pixels bruts

In [ ]:
# TODO : projeter X_train_sc et X_test_sc sur 50 composantes ACP
# Entraîner un arbre (max_depth=10) sur les pixels bruts ET sur les features ACP
# Comparer accuracy et temps d'entraînement

pca_50 = PCA(n_components=50)
X_train_50 = pca_50.fit_transform(X_train_sc)
X_test_50  = pca_50.transform(X_test_sc)

# Arbre sur pixels bruts
t0 = time.time()
# Votre code ici
t_raw = time.time() - t0

# Arbre sur ACP
t0 = time.time()
# Votre code ici
t_pca = time.time() - t0

print("=== Arbre de décision (max_depth=10) ===")
print(f"Pixels bruts (784 feat.) — Accuracy test : ...  | Temps : {t_raw:.2f}s")
print(f"ACP 50 composantes       — Accuracy test : ...  | Temps : {t_pca:.2f}s")

### 📝 Observations

*À quelle profondeur l'arbre commence-t-il à sur-apprendre ?  
La réduction ACP améliore-t-elle l'accuracy ou seulement la vitesse ?  
Quelle est la limite fondamentale d'un seul arbre de décision ?*



---

# Partie 4 — Méthodes d'Ensemble : Random Forest et AdaBoost

## Rappel théorique

### Bagging → Random Forest
Le **bagging** (Bootstrap AGGregatING) entraîne $T$ arbres sur des **sous-échantillons bootstrap** et agrège leurs prédictions par vote majoritaire.  
Le **Random Forest** ajoute une sélection **aléatoire d'un sous-ensemble de features** à chaque coupure, réduisant la corrélation entre arbres.

$$\hat{y} = \text{vote}\left(h_1(x), h_2(x), \ldots, h_T(x)\right)$$

### Boosting → AdaBoost
Le **boosting** entraîne les arbres **séquentiellement** : chaque arbre se concentre sur les exemples mal classés par le précédent via une réévaluation des poids.

$$F_T(x) = \sum_{t=1}^{T} \alpha_t h_t(x)$$

où $\alpha_t$ est le poids de l'arbre $t$ selon son taux d'erreur.

## Étape 1 — Random Forest sur features ACP

In [ ]:
# TODO : entraîner un RandomForestClassifier (n_estimators=100, random_state=42)
# sur X_train_50 (ACP 50 composantes)
# Afficher l'accuracy train et test, et le temps d'entraînement

t0 = time.time()
rf = ...
# Votre code ici
t_rf = time.time() - t0

print(f"Random Forest — Accuracy train : ...  | test : ...  | Temps : {t_rf:.2f}s")

## Étape 2 — Effet du nombre d'arbres (n_estimators)

In [ ]:
# TODO : entraîner des Random Forests avec n_estimators dans [1, 5, 10, 25, 50, 100, 200]
# Tracer accuracy test en fonction du nombre d'arbres

n_estimators_list = [1, 5, 10, 25, 50, 100, 200]
rf_acc_test = []

for n in n_estimators_list:
    # Votre code ici
    pass

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_list, rf_acc_test, 'o-', color='green')
plt.xlabel("Nombre d'arbres (n_estimators)")
plt.ylabel("Accuracy test")
plt.title("Random Forest — Accuracy selon le nombre d'arbres")
plt.tight_layout()
plt.show()

## Étape 3 — AdaBoost sur features ACP

In [ ]:
# TODO : entraîner un AdaBoostClassifier (n_estimators=100, random_state=42)
# sur X_train_50
# Afficher l'accuracy train et test, et le temps d'entraînement
# ⚠️ AdaBoost peut être lent sur MNIST — c'est normal

t0 = time.time()
ada = ...
# Votre code ici
t_ada = time.time() - t0

print(f"AdaBoost — Accuracy train : ...  | test : ...  | Temps : {t_ada:.2f}s")

## Étape 4 — Tableau comparatif de tous les modèles

In [ ]:
# TODO : remplir ce tableau récapitulatif avec les résultats obtenus dans les parties précédentes
# Calcul de la matrice de confusion pour le meilleur modèle

import pandas as pd

# TODO : compléter les valeurs (...) avec vos résultats
results = pd.DataFrame({
    'Modèle': ['Arbre (max_depth=10, raw)', 'Arbre (max_depth=10, ACP-50)',
               'Random Forest (100 arbres, ACP-50)', 'AdaBoost (100 estim., ACP-50)'],
    'Accuracy Test': [..., ..., ..., ...],
    'Temps (s)':     [..., ..., ..., ...]
})

print(results.to_string(index=False))

# Matrice de confusion du meilleur modèle
# TODO : choisir le meilleur modèle et afficher sa matrice de confusion
best_model = rf  # à modifier si un autre modèle est meilleur
y_pred_best = best_model.predict(X_test_50)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title("Matrice de confusion — Meilleur modèle (test)")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.tight_layout()
plt.show()

### 📝 Observations

*Quel modèle offre le meilleur compromis accuracy / temps ?  
Quels chiffres sont le plus souvent confondus ?  
Pourquoi le Random Forest est-il généralement plus robuste qu'un seul arbre ?  
Quelle est la différence fondamentale entre bagging et boosting ?*



---

# Partie 5 — ⭐ Bonus : Pipeline Complet et GridSearchCV

## Objectif

Dans cette partie bonus, vous allez construire un **pipeline sklearn complet** combinant :
- `StandardScaler`
- `PCA` (nombre de composantes à optimiser)
- Un classifieur (à choisir)

Et utiliser **GridSearchCV** pour tester simultanément plusieurs hyperparamètres — y compris le nombre de composantes ACP.

> 💡 Le nom des hyperparamètres dans un Pipeline suit la convention `etape__parametre` (double underscore).  
> Exemple : `pca__n_components` pour le `n_components` de l'étape nommée `'pca'`.

## Étape 1 — Construction du Pipeline

In [ ]:
# TODO : construire un Pipeline avec les étapes suivantes :
# 1. 'scaler' : StandardScaler()
# 2. 'pca'    : PCA()
# 3. 'clf'    : DecisionTreeClassifier(random_state=42)  ← vous pourrez changer de classifieur

pipeline = Pipeline([
    # Votre code ici
])

print(pipeline)

## Étape 2 — GridSearchCV sur ACP + Arbre de Décision

In [ ]:
# TODO : définir la grille de paramètres et lancer GridSearchCV
# ⚠️ Utilisez cv=3 et un sous-ensemble de X_train pour limiter le temps de calcul

# Sous-ensemble pour la rapidité
X_gs, _, y_gs, _ = train_test_split(X_train_sc, y_train, train_size=2000, random_state=42, stratify=y_train)

param_grid = {
    'pca__n_components': [20, 50, 100],
    'clf__max_depth':    [5, 10, 15],
    'clf__criterion':   ['gini', 'entropy']
}

# TODO : instancier GridSearchCV et l'ajuster
grid_search = ...
# Votre code ici

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur score CV    : {grid_search.best_score_:.4f}")

# Évaluer le meilleur modèle sur le test complet
best_pipe = grid_search.best_estimator_
print(f"Accuracy test (meilleur pipeline) : {best_pipe.score(X_test_sc, y_test):.4f}")

## Étape 3 — Exploration libre : testez votre propre grille

Remplacez l'arbre de décision par un **Random Forest** et relancez le GridSearchCV avec votre propre grille d'hyperparamètres.  
**Contrainte** : budget maximal de **50 composantes ACP**.

In [ ]:
# TODO : construire un nouveau pipeline avec RandomForestClassifier
# Définir votre propre param_grid (n_components <= 50, n_estimators, max_depth...)
# Lancer GridSearchCV et commenter vos résultats

# Votre code ici


### 📝 Observations

*Quelle combinaison (n_components, hyperparamètres) donne les meilleures performances ?  
La recherche sur la grille est-elle coûteuse en temps ? Pourquoi ?  
Quelles alternatives à GridSearchCV pourriez-vous envisager pour un espace de recherche plus grand ?*



---

## Résumé — Points Clés

| Concept | Point clé |
|---|---|
| ACP | Réduction de 784 → quelques dizaines de composantes sans perte majeure d'information |
| Eigendigits | Les composantes principales capturent les modes de variation visuels |
| PCR | ACP + régression — contrôle la multicolinéarité, k choisi par CV |
| Arbre de décision | Interprétable mais sur-apprend facilement (max_depth critique) |
| Random Forest | Bagging d'arbres → variance réduite, plus robuste qu'un seul arbre |
| AdaBoost | Boosting séquentiel → biais réduit, sensible aux outliers |
| Pipeline sklearn | Chaîne reproductible et compatible avec GridSearchCV |
| GridSearchCV | Recherche automatique des hyperparamètres optimaux par validation croisée |